# Segunda fase

In [19]:
import pandas

# Importing classes of the project

# Reload classes in memory every time this code block is executed
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from data_analysis.analizer import DataAnalizer

from model_building.KNN import KNN
from model_building.DecisionTree import DecisionTree
from model_building.ModelEvaluator import ModelEvaluator


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
df = pd.read_csv("../out/dataset.csv")
del df["congestion_surcharge"]
del df["pickup_time_in_seconds"]
del df["dropoff_time_in_seconds"]

# Remove all rows with any missing values (NaN)
df= df.dropna()
df


,trip_distance,fare_amount,tip_amount,tolls_amount,extra,passenger_count,pickup_hour,pickup_day_of_week,pickup_day_of_month,pickup_month,dropoff_hour,dropoff_day_of_week,dropoff_day_of_month,dropoff_month,mta_tax,vendorid,ratecodeid,pulocationid,dolocationid,payment_type
0,0.38,3.5,0.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,2.0,1.0,170.0,170.0,2.0
1,1.40,6.5,4.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,1.0,1.0,229.0,141.0,1.0
2,1.20,7.0,1.70,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,1.0,1.0,144.0,158.0,1.0
3,2.39,10.0,0.00,0.0,0.5,1.0,0,1,1,1,0,1,1,1,0.5,2.0,1.0,244.0,69.0,2.0
4,9.44,28.0,5.00,0.0,0.5,1.0,0,1,1,1,1,1,1,1,0.5,2.0,1.0,114.0,42.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84564,1.22,7.0,3.24,0.0,0.5,1.0,23,1,31,12,23,1,31,12,0.5,2.0,1.0,90.0,68.0,1.0
84565,0.40,4.0,1.55,0.0,3.0,2.0,23,1,31,12,23,1,31,12,0.5,1.0,1.0,79.0,107.0,1.0
84566,1.48,7.5,3.39,0.0,0.5,1.0,23,1,31,12,23,1,31,12,0.5,2.0,1.0,161.0,234.0,1.0
84567,0.90,5.5,0.00,0.0,0.5,5.0,23,1,31,12,22,2,1,1,0.5,2.0,1.0,68.0,246.0,2.0


In [4]:
from sklearn.preprocessing import StandardScaler



# =============================================
# 1. REGRESSION DATASET (continuous target)
# =============================================

# Separate features and target
X_reg = df.drop(columns=['fare_amount'])
y_reg = df['fare_amount']

# Scale only the features (not target)
scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

# Create scaled DataFrame for regression
df_regression = pd.DataFrame(X_reg_scaled, columns=X_reg.columns)
df_regression['fare_amount'] = y_reg.values  # Add unscaled target

# =============================================
# 2. CLASSIFICATION DATASET (categorical target)
# =============================================

# Create fare classes
bins = [-np.inf, 10, 30, 60, np.inf]
labels = [1, 2, 3, 4]

# Create classification target
df_classification = df.copy()
df_classification['fare_class'] = pd.cut(
    df['fare_amount'],
    bins=bins,
    labels=labels
)

# Separate features and target
X_clf = df_classification.drop(columns=['fare_amount', 'fare_class'])
y_clf = df_classification['fare_class']

# Scale features using SAME scaler (important for consistency)
X_clf_scaled = scaler.transform(X_clf)  # Use existing scaler

# Create scaled DataFrame for classification
df_classification_scaled = pd.DataFrame(X_clf_scaled, columns=X_clf.columns)
df_classification_scaled['fare_class'] = y_clf.values  # Add target

analizer_reg = DataAnalizer(df_regression, "fare_amount", test_size=0.02)

# =============================================
# Verification
# =============================================
print("Regression dataset:")
print(df_regression.head())

print("\nClassification dataset:")
print(df_classification_scaled.head())

print(df_classification['fare_class'].value_counts(normalize=True))

analizer_clf = DataAnalizer(df_classification_scaled, "fare_class", test_size=0.2)

Data divided successfully.
Regression dataset:
   trip_distance  tip_amount  tolls_amount     extra  passenger_count  \
0      -0.666634   -0.760919     -0.229742 -0.463238        -0.470065   
1      -0.404075    0.619532     -0.229742 -0.463238        -0.470065   
2      -0.455557   -0.174228     -0.229742 -0.463238        -0.470065   
3      -0.149237   -0.760919     -0.229742 -0.463238        -0.470065   
4       1.665513    0.964644     -0.229742 -0.463238        -0.470065   

   pickup_hour  pickup_day_of_week  pickup_day_of_month  pickup_month  \
0    -2.316139           -1.018927            -1.674574     -1.532716   
1    -2.316139           -1.018927            -1.674574     -1.532716   
2    -2.316139           -1.018927            -1.674574     -1.532716   
3    -2.316139           -1.018927            -1.674574     -1.532716   
4    -2.316139           -1.018927            -1.674574     -1.532716   

   dropoff_hour  dropoff_day_of_week  dropoff_day_of_month  dropoff_month  

# KNN

O KNN (K-Nearest Neighbors, ou K-Vizinhos Mais Próximos) é um algoritmo de aprendizado de máquina supervisionado usado para classificação e regressão. Ele se baseia no princípio de que objetos semelhantes estão próximos no espaço de características.

**Funcionamento:**


Calcula a distância (ex.: Euclidiana, Manhattan) entre o novo dado e todos os pontos no conjunto de treinamento.

Seleciona os K vizinhos mais próximos.

Classifica (moda das classes dos vizinhos) ou prediz (média dos valores dos vizinhos).



In [26]:
# Regression Analysis of KNN
rmse_results = {}
for i in range(2, 26, 5):
    print(f"Training KNN Regression for k = {i}")
    knn_reg = KNN(i, task="regression", verbose=True)
    knn_reg.fit(analizer_reg.data_train, analizer_reg.labels_train)
    pred_values = knn_reg.predict(analizer_reg.data_test)

    # Calculate Root Mean Squared Error
    rmse = np.sqrt(np.mean((pred_values - analizer_reg.labels_test)**2))
    rmse_results[i] = rmse
    print(f"k={i}: RMSE = {rmse:.2f}")

print("\nFinal Regression Results:")
print(rmse_results)

Data divided successfully.
Training KNN Regression for k = 5
KD-Tree built with 131068 nodes, dimensions=19
Processed 1000/1688
Prediction complete
k=5: RMSE = 4.43
Training KNN Regression for k = 10
KD-Tree built with 131068 nodes, dimensions=19
Processed 1000/1688
Prediction complete
k=10: RMSE = 4.63
Training KNN Regression for k = 15
KD-Tree built with 131068 nodes, dimensions=19
Processed 1000/1688
Prediction complete
k=15: RMSE = 4.80
Training KNN Regression for k = 20
KD-Tree built with 131068 nodes, dimensions=19
Processed 1000/1688
Prediction complete
k=20: RMSE = 4.90
Training KNN Regression for k = 25
KD-Tree built with 131068 nodes, dimensions=19
Processed 1000/1688
Prediction complete
k=25: RMSE = 4.99

Final Regression Results:
{5: np.float64(4.427841938487359), 10: np.float64(4.626283588500842), 15: np.float64(4.79722290519828), 20: np.float64(4.8972478527938845), 25: np.float64(4.990911051727522)}


In [ ]:
# Classification Analysis of KNN
precision_results = {}
for i in range(5, 26, 5):
    print(f"Training KNN Classification for k = {i}")
    knn_clf = KNN(i, task="classification", verbose=True)
    knn_clf.fit(analizer_clf.data_train, analizer_clf.labels_train)
    pred_labels = knn_clf.predict(analizer_clf.data_test)

    # Calculate Precision (accuracy)
    precision = np.mean(pred_labels == analizer_clf.labels_test)
    precision_results[i] = precision
    print(f"k={i}: Precision = {precision:.4f}")

print("\nFinal Classification Results:")
print(precision_results)

Data divided successfully.
Training KNN Classification for k = 5
KD-Tree built with 131008 nodes, dimensions=19
Processed 1000/16876
Processed 2000/16876
Processed 3000/16876
Processed 4000/16876
Processed 5000/16876
Processed 6000/16876
Processed 7000/16876
Processed 8000/16876
Processed 9000/16876
Processed 10000/16876
Processed 11000/16876
Processed 12000/16876
Processed 13000/16876
Processed 14000/16876
Processed 15000/16876
Processed 16000/16876
Prediction complete
k=5: Precision = 0.7533
Training KNN Classification for k = 10
KD-Tree built with 131008 nodes, dimensions=19
Processed 1000/16876
Processed 2000/16876
Processed 3000/16876
Processed 4000/16876
Processed 5000/16876
Processed 6000/16876
Processed 7000/16876
Processed 8000/16876
Processed 9000/16876
Processed 10000/16876
Processed 11000/16876
Processed 12000/16876
Processed 13000/16876
Processed 14000/16876
Processed 15000/16876
Processed 16000/16876
Prediction complete
k=10: Precision = 0.7489
Training KNN Classification

# Árvore de decisão

A Árvore de Decisão é um algoritmo de aprendizado de máquina supervisionado usado para classificação e regressão. Ele divide os dados em subconjuntos com base em regras de decisão hierárquicas, formando uma estrutura semelhante a uma árvore.

**Funcionamento:**

Seleciona o melhor atributo para dividir os dados (usando critérios como Gini, Entropia ou Erro quadrático).

Divide o dataset recursivamente, criando nós de decisão até atingir uma condição de parada (ex.: profundidade máxima ou número mínimo de amostras por folha).

Classifica ou prediz com base na folha (nó final) em que o dado cai.

In [20]:
# Classification with Decision Tree
for depth in [None, 2, 5, 10, 20]:
    print(f"Training Decision Tree Classification for max_depth = {depth}")
    dt_clf = DecisionTree(
        problem_type='classification',
        max_depth=depth,
        class_weight='balanced'
    )

    # evaluate the model
    evaluator = ModelEvaluator(
        model=dt_clf,
        task_type="multiclass",
        metrics=["accuracy", "precision_macro", "recall_macro", "f1_macro", "log_loss"],
        cv_method='stratifiedkfold',
        cv_folds=5,
    )

    results_clf = evaluator.evaluate(analizer_clf.data_train, analizer_clf.labels_train)
    print("=== Classificação (Decision Tree) ===")
    print(f"max_depth={depth}")
    for metric, value in results_clf.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")



Training Decision Tree Classification for max_depth = None
=== Classificação (Decision Tree) ===
max_depth=None
accuracy_mean: 0.8909
precision_macro_mean: 0.8647
recall_macro_mean: 0.8551
f1_macro_mean: 0.8597
log_loss_mean: 3.9306
accuracy_std: 0.0026
precision_macro_std: 0.0063
recall_macro_std: 0.0055
f1_macro_std: 0.0047
log_loss_std: 0.0941
Training Decision Tree Classification for max_depth = 2
=== Classificação (Decision Tree) ===
max_depth=2
accuracy_mean: 0.8176
precision_macro_mean: 0.7384
recall_macro_mean: 0.7463
f1_macro_mean: 0.7356
log_loss_mean: 0.6380
accuracy_std: 0.0971
precision_macro_std: 0.1310
recall_macro_std: 0.0586
f1_macro_std: 0.0984
log_loss_std: 0.0639
Training Decision Tree Classification for max_depth = 5
=== Classificação (Decision Tree) ===
max_depth=5
accuracy_mean: 0.8586
precision_macro_mean: 0.7579
recall_macro_mean: 0.8835
f1_macro_mean: 0.8055
log_loss_mean: 0.3507
accuracy_std: 0.0034
precision_macro_std: 0.0230
recall_macro_std: 0.0098
f1_macr

In [21]:
# Regression with Decision Tree

regression_results_cv = {}
for depth in [None, 2, 5, 10, 20]:
    print(f"Training Decision Tree Regression for max_depth = {depth}")
    dt_reg = DecisionTree(problem_type='regression', max_depth=depth, random_state=42)

    # Evaluate the model using ModelEvaluator for regression with multiple metrics
    evaluator_reg = ModelEvaluator(
        model=dt_reg,
        task_type="regression",
        metrics=['rmse', 'mse', 'mae', 'r2'],  # Evaluate using RMSE, MSE, MAE, and R-squared
        cv_method='kfold',
        cv_folds=5,
        random_state=42
    )

    results_reg = evaluator_reg.evaluate(analizer_reg.data_train, analizer_reg.labels_train)

    regression_results_cv[depth] = {
        'rmse_mean': results_reg.get('rmse_mean'),
        'rmse_std': results_reg.get('rmse_std'),
        'mse_mean': results_reg.get('mse_mean'),
        'mse_std': results_reg.get('mse_std'),
        'mae_mean': results_reg.get('mae_mean'),
        'mae_std': results_reg.get('mae_std'),
        'r2_mean': results_reg.get('r2_mean'),
        'r2_std': results_reg.get('r2_std')
    }

    print("=== Regressão (Decision Tree) ===")
    print(f"max_depth={depth}")
    for metric, value in regression_results_cv[depth].items():
        if value is not None:
            print(f"{metric}: {value:.2f}")
    print("-" * 30)

print("\nFinal Regression Results (Cross-Validation):")
print(regression_results_cv)

Training Decision Tree Regression for max_depth = None
=== Regressão (Decision Tree) ===
max_depth=None
rmse_mean: 5.52
rmse_std: 1.68
mse_mean: 33.34
mse_std: 21.70
mae_mean: 1.69
mae_std: 0.05
r2_mean: 0.80
r2_std: 0.10
------------------------------
Training Decision Tree Regression for max_depth = 2
=== Regressão (Decision Tree) ===
max_depth=2
rmse_mean: 7.25
rmse_std: 1.16
mse_mean: 53.91
mse_std: 18.71
mae_mean: 3.26
mae_std: 0.04
r2_mean: 0.66
r2_std: 0.07
------------------------------
Training Decision Tree Regression for max_depth = 5
=== Regressão (Decision Tree) ===
max_depth=5
rmse_mean: 6.25
rmse_std: 1.80
mse_mean: 42.26
mse_std: 23.55
mae_mean: 2.01
mae_std: 0.04
r2_mean: 0.74
r2_std: 0.13
------------------------------
Training Decision Tree Regression for max_depth = 10
=== Regressão (Decision Tree) ===
max_depth=10
rmse_mean: 5.90
rmse_std: 2.00
mse_mean: 38.81
mse_std: 24.59
mae_mean: 1.58
mae_std: 0.04
r2_mean: 0.76
r2_std: 0.14
------------------------------
Trai

# Support Vector Machine (SVM)

O SVM (Máquina de Vetores de Suporte) é um algoritmo de aprendizado de máquina supervisionado usado para classificação e regressão. Ele busca encontrar o hiperplano ótimo que melhor separa diferentes classes no espaço de características, maximizando a margem entre os pontos mais próximos (vetores de suporte).

**Funcionamento:**

Mapeia os dados para um espaço de maior dimensão** (usando kernels como linear, polinomial ou RBF, se necessário).

Encontra o hiperplano com a maior margem de separação entre classes.

Classifica novos dados com base em qual lado do hiperplano eles estão.

In [ ]:
# Regression with SVM

In [ ]:
# Classification with SVM